<a href="https://colab.research.google.com/github/SashaNasonova/snowStudy/blob/main/gee_sentinel2_albedo_ndsi_ndwi_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Snow Study Notebook

This notebook was created to generate albedo, ndsi and ndwi rasters for a time-series albedo study that aims to track albedo changes through time. The methods are based on Hammaer et al, 2023 and are as follows:

- Search Sentinel-2 Surface Reflectance archive
- Mask clouds using s2cloudless (40% probability, 0.15 NIR, 50m buffer, max distance of 2 km from cloud edge for cloud shadow search)
- Bands: blue - b2, green - b3, red - b4, NIR - b8, swir1 - b11, swir2 - b12
- Albedo =  0.356b2  + 0.130b4 + 0.373b8 + 0.085b11 + 0.072b12 - 0.0018
- NDSI = (b3 - b11) / (b3 + b11)
- NDWI = (b3 - b8)/ (b3 + b8)
- Export cloud raster (1 for cloud)
- Export truecolor/fcir 3-band images for visualization


In [ ]:
#Install libraries
%pip install geemap
%pip install pycrs rasterio python-pptx cartopy requests

In [ ]:
#Import libraries
import os, shutil
import ee, geemap
import geopandas
from osgeo import gdal
from google.colab import files

### Define functions
Define albedo, ndsi, ndwi and cloud masking functions here.

In [ ]:
## Define functions
# Helper function to get files, non-recursive
def getfiles(d,ext):
  paths = []
  for file in os.listdir(d):
      if file.endswith(ext):
          paths.append(os.path.join(d, file))
  return(paths)

# Helper function to get image acquisition date and format into ("yyyy-mm-dd")
def getDate(im):
  return(ee.Image(im).date().format("YYYY-MM-dd"))

# Helper function to get scene ids
def getSceneIds(im):
  return(ee.Image(im).get('PRODUCT_ID'))

# Functions to mosaic by image date
def mosaicByDate(indate):
  d = ee.Date(indate)
  #print(d)
  im = col.filterBounds(poly).filterDate(d, d.advance(1, "day")).mosaic()
  #print(im)
  return(im.set("system:time_start", d.millis(), "system:index", d.format("YYYY-MM-dd")))

def runDateMosaic(col_list):
  #get a list of unique dates within the list
  date_list = col_list.map(getDate).getInfo()
  udates = list(set(date_list))
  udates.sort()
  udates_ee = ee.List(udates)

  #mosaic images by unique date
  mosaic_imlist = udates_ee.map(mosaicByDate)
  return(ee.ImageCollection(mosaic_imlist))

# Multiplies Sentinel-2 imagery by 0.0001
def apply_scale_factors_s2(image):
  opticalBands = image.select('B.*').multiply(0.0001)
  return image.addBands(opticalBands, None, True)

# Calculate NDSI (Sentinel-2)
def NDSI_S2(image):
  ndsi = image.expression(
      '(Green - SWIR1) / (Green + SWIR1)', {
          'Green': image.select('B3'),
          'SWIR1': image.select('B11')}).rename('ndsi')
  return(ndsi)

# Calculate NDWI (Sentinel-2)
def NDWI_S2(image):
  ndwi = image.expression(
      '(Green - NIR) / (Green + NIR)', {
          'Green': image.select('B3'),
          'NIR': image.select('B8')}).rename('ndwi')
  return(ndwi)

# Calculate albedo (Sentinel-2), this will be 20m because of SWIR bands
def albedo_S2(image):
  #0.356b2 + 0.130b4 + 0.373b8 + 0.085b11 + 0.072b12 - 0.0018
  albedo = image.expression(
      '0.356*Blue + 0.130*Red + 0.373*NIR + 0.085*SWIR1 + 0.072SWIR2 - 0.0018', {
          'Blue': image.select('B2'),
          'Red': image.select('B4'),
          'NIR': image.select('B8'),
          'SWIR1':image.select('B11'),
          'SWIR2': image.select('B12')}).rename('albedo')
  return(albedo)

# Tiling function, uses a geometry (footprint) to split into a defined
# number or rows and columns (nx,ny)
def grid_footprint(footprint,nx,ny):
  from shapely.geometry import Polygon, LineString, MultiPolygon
  from shapely.ops import split

  #polygon = footprint
  polygon = Polygon(footprint['coordinates'][0])
  #polygon = Polygon(footprint)

  minx, miny, maxx, maxy = polygon.bounds
  dx = (maxx - minx) / nx  # width of a small part
  dy = (maxy - miny) / ny  # height of a small part

  horizontal_splitters = [LineString([(minx, miny + i*dy), (maxx, miny + i*dy)]) for i in range(ny)]
  vertical_splitters = [LineString([(minx + i*dx, miny), (minx + i*dx, maxy)]) for i in range(nx)]
  splitters = horizontal_splitters + vertical_splitters

  result = polygon
  for splitter in splitters:
      result = MultiPolygon(split(result, splitter))

  coord_list = [list(part.exterior.coords) for part in result.geoms]

  poly_list = []
  for cc in coord_list:
      p = ee.Geometry.Polygon(cc)
      poly_list.append(p)
  return(poly_list)

def aoionly(img):
  return(img.updateMask(poly_mask))

###Cloud masking functions for s2cloudless
###(https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless)

# Get surface reflectance data and corresponding s2cloudless dataset, merge
def get_s2_sr_cld_col(aoi, start_date, end_date,CLOUD_FILTER):
    # Import and filter S2 SR.
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER)))

    # Import and filter s2cloudless.
    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
                        .filterBounds(aoi)
                        .filterDate(start_date, end_date))

    # Join the filtered s2cloudless collection to the SR collection by the 'system:index' property.
    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))

# Get cloud bands
# Cloud probability is defined in the main body of the script, using 50%, default is 30% but I found that too aggressive
def add_cloud_bands(img):
    # Get s2cloudless image, subset the probability band.
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')

    # Condition s2cloudless by the probability threshold value.
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')

    # Add the cloud probability layer and cloud mask as image bands.
    return img.addBands(ee.Image([cld_prb, is_cloud]))

# Create cloud shadow bands
# NIR_DRK_THRESH is defined in the main body of the script
def add_shadow_bands(img):
    # Identify water pixels from the SCL band.
    not_water = img.select('SCL').neq(6)

    # Identify dark NIR pixels that are not water (potential cloud shadow pixels).
    SR_BAND_SCALE = 1e4
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE).multiply(not_water).rename('dark_pixels')

    # Determine the direction to project cloud shadow from clouds (assumes UTM projection).
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')));

    # Project shadows from clouds for the distance specified by the CLD_PRJ_DIST input.
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    # Identify the intersection of dark pixels with cloud shadow projection.
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')

    # Add dark pixels, cloud projection, and identified shadows as image bands.
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

# Merge cloud and cloud shadow mask, clean up
# BUFFER is defined in the main body of the script, default 50m, I use 10m. Again, too aggressive
# There is confusion between snow and cloud

def add_cld_shdw_mask(img):
    # Add cloud component bands.
    img_cloud = add_cloud_bands(img)

    # Add cloud shadow component bands.
    img_cloud_shadow = add_shadow_bands(img_cloud)

    # Combine cloud and shadow mask, set cloud and shadow as value 1, else 0.
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)

    # Remove small cloud-shadow patches and dilate remaining pixels by BUFFER input.
    # 20 m scale is for speed, and assumes clouds don't require 10 m precision.
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))

    # Add the final cloud-shadow mask to the image.
    return img_cloud_shadow.addBands(is_cld_shdw)

# Apply all masks
def apply_cld_shdw_mask(img):
    # Subset the cloudmask band and invert it so clouds/shadow are 0, else 1.
    not_cld_shdw = img.select('cloudmask').Not()

    # Subset reflectance bands and update their masks, return the result.
    return img.select('B.*').updateMask(not_cld_shdw)

In [ ]:
# Authenticate gee
ee.Authenticate()

In [ ]:
# Initialize with google cloud project
project = 'snow-study'
ee.Initialize(project=project)

In [ ]:
# Download area of interest from BCBox (make sure it's set to public)
# Open fires shapefile if exists
aoi_shp = '/content/sf_poly_sub.shp'

if os.path.exists(aoi_shp):
  print(aoi_shp,'exists')
else:
  raise ValueError(f"{aoi_shp} does not exist")

In [ ]:
# Visualize table and select polygon
import warnings
from google.colab import data_table
warnings.filterwarnings("ignore")
data_table.enable_dataframe_formatter()

# Visualize in table format
poly = geemap.shp_to_ee(aoi_shp)
poly_df = geopandas.read_file(aoi_shp)
poly_df_tbl = poly_df.drop(columns=['geometry'], axis=1, inplace=False)
poly_df_tbl

In [ ]:
# Add to map to visualize
Map = geemap.Map()
Map.addLayer(poly,{},'Area of Interest')
Map.centerObject(poly,zoom=12)
Map

In [ ]:
# Create output folder
aoi_str = 'sf_poly_sub'

if not os.path.exists(aoi_str):
  os.mkdir(aoi_str)

# Save a copy of the polygon in a vectors folder
vector_folder = os.path.join(aoi_str,'vectors')
if not os.path.exists(vector_folder):
  os.mkdir(vector_folder)
out_shp = os.path.join(vector_folder,aoi_str+'.shp')
poly_df.to_file(out_shp,driver='ESRI Shapefile')

# Create raster mask to reduce extent of image collections
# Function aoionly in functions
poly_buf = poly.geometry().buffer(500).bounds()
poly_mask = ee.Image.constant(1).clip(poly_buf).selfMask()

In [ ]:
# Select collection, Sentinel-2 Surface Reflectance. Flexible to add other data sources.
dattype = 'S2'
cld_field =  'CLOUDY_PIXEL_PERCENTAGE'
col_name = 'COPERNICUS/S2_SR_HARMONIZED'

col = ee.ImageCollection(col_name)
print(col_name,'selected')

In [ ]:
# Search imagery and cloud mask (Sentinel-2 only)
startdate = '2023-09-01'
enddate = '2023-10-01'
CLOUD_FILTER = 70
CLD_PRB_THRESH = 50
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 10

s2_sr_col = col.filterDate(startdate,enddate).filterBounds(poly).filter(ee.Filter.lt(cld_field,CLOUD_FILTER)).map(aoionly)

col = get_s2_sr_cld_col(poly, START_DATE, END_DATE)
col_wmsks = col.map(add_cld_shdw_mask)
s2_noclouds = col_wmsks.map(apply_cld_shdw_mask)

col_filtered_size = col_filtered.size().getInfo() #get size of the collection
col_filtered_list = col_filtered.toList(col_filtered_size)
col_filtered_mosaic = runDateMosaic(col_filtered_list)
col_mosaic = col_filtered_mosaic.map(apply_scale_factors_s2)

print('Found',col_filtered_size,'images')

In [ ]:
# Calculate indices and albedo
ndsi = col_mosaic.map(NDSI_S2)
ndwi = col_mosaic.map(NDWI_S2)
albedo = col_mosaic.map(albedo_S2)

In [ ]:
# Export rasters: ndwi, ndsi, albedo, truecolor rgb, fcir rgb, cloud
crs_out = 'EPSG:3005'
outfolder = aoi_str
px_ind = 20
px_rgb = 10

folders = [
    'ndsi',
    'ndwi',
    'albedo',
    'truecolor',
    'falsecolor']

for folder_name in folders:
    folder_path = os.path.join(outfolder, folder_name)
    if os.path.exists(folder_path):
        print(f"{folder_path} exists. Deleting.")
        shutil.rmtree(folder_path)
    os.makedirs(folder_path)

## Define tiling rules
poly_area = round(poly.geometry().area(1).divide(10000).getInfo(),1)
print(poly_area)
print('Poly Area:',poly_area,'hectares')

if poly_area < 10000:
    n = 2
elif poly_area >= 10000 and poly_area < 100000:
    n = 3
elif poly_area >= 100000 and poly_area < 400000:
    n = 4
elif poly_area >= 400000 and poly_area < 1000000:
    n = 5
else:
    n = 6

print('Number of tiles: ' + str(n*n))

## Create tiles
footprint = poly.geometry().bounds().getInfo()
grids = grid_footprint(footprint,n,n)

## Export NDSI
for i in range(0,len(grids)):
  roi = grids[i]
  print(roi)

  ## Export NDSI
  geemap.ee_export_image_collection(ndsi,out_dir=os.path.join(outfolder,'ndsi',str(i)), scale=px_ind, region=roi,file_per_band=True)

  ## Export NDWI

  ## Export truecolor 3-band image

  ## Export fcir 3-band image


In [ ]:
def getfilenames(d,ext):
  paths = []
  for file in os.listdir(d):
      if file.endswith(ext):
          paths.append(file)
  return(paths)

def getfiles_recursive(d, filename):
    paths = []
    for root, dirs, files in os.walk(d):
        for file in files:
            if file==filename:
                paths.append(os.path.join(root, file))
    return paths

def mosaic_all(filelist,outpath):
  gdal.Warp(outpath,filelist)
  for file in filelist: os.remove(file) #delete tiles

  #clean up empty folders
  path = os.path.dirname(outpath)
  for root, dirs, files in os.walk(path, topdown=False):
        for name in dirs:
            full_path = os.path.join(root, name)
            # Check if the directory is empty
            if not os.listdir(full_path):
                os.rmdir(full_path)
                print(f"Removed empty folder: {full_path}")
  print('mosaicking complete')

# Create master list of dates
ndsi_0 = os.path.join(outfolder,'ndsi','0')
master_list = getfilenames(ndsi_0,'.tif')
print(master_list)

# Recursively get all filepaths inside ndsi subfolder and mosaic
ndsi_folder = os.path.join(outfolder,'ndsi')

for m in master_list:
  print(m)
  ndsi_list = getfiles_recursive(ndsi_folder,m)
  mosaic_all(ndsi_list,os.path.join(ndsi_folder,m))

# print(ndsi_list)

# Recursively get all filepaths inside ndwi subfolder and mosaic

# Recursively get all filepaths inside tc subfolder and mosaic
# Recursively get all filepaths inside fcir subfolder and mosaic

In [ ]:
#Zip to download
zipped = outfolder + '.zip'

!zip -r {zipped} {outfolder}
files.download(zipped)